### imports y creo DF

In [ ]:
from graphviz import Digraph
import numpy as np
import pandas as pd

from datetime import datetime, timedelta


file_path = 'bases/para tesis LCC_Datos.xlsx'
df_totales = pd.read_excel(file_path)
df_planes_por_alumno = pd.read_excel(file_path, sheet_name="planes por alumno")
df_cursadas_por_alumno = pd.read_excel(file_path, sheet_name="cursadas por alumno (2)") #pd.read_excel(file_path, sheet_name="cursadas por alumno")
df_lista_de_materias = pd.read_excel(file_path, sheet_name="lista de materias")
df_finales_por_alumno = pd.read_excel(file_path, sheet_name="finales por alumno")
df_estados_de_actas = pd.read_excel(file_path, sheet_name="estados de actas")
df_resultados_posibles = pd.read_excel(file_path, sheet_name="resultados posibles")


In [ ]:
df_planes_por_alumno

In [ ]:
# df_materias_dc = 
# df_lista_de_materias[df_lista_de_materias['codigo_materia'].str.contains('COMP93')].sort_values(by="codigo_materia")
# nodo = np.zeros(shape=(1,len(df_materias_dc)))
# df_lista_de_materias[df_lista_de_materias['codigo_materia'].str.contains('LCDA21')].sort_values(by="codigo_materia")[:11]

#[LCDA210001 : LCDA210010] U 
# Algos: ["COMP930003","COMP930004","COMP930005"]
# Algebra COMP930001, LCDA210002

In [ ]:
df_lista_de_materias[df_lista_de_materias['codigo_materia'].str.contains('COMP23')].sort_values(by="codigo_materia") 
# nueva Algo : "COMP230008", "COMP230001", "COMP230015"

___________________________
### Filtro de "cursadas" de YEAR, del primer cuatri/bimestre, y que se haya inscripto en materias de Álgebra I (de lcd o compu viejo ) ó Algo1 (compu nuevo o viejo)

In [ ]:
year=2021

In [ ]:
cursaronYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['anio_academico'] == year) 
                                     & (df_cursadas_por_alumno['periodo'].isin(['1er Cuatrimestre', '1er Bimestre'])) 
                                     & (df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'LCDA210002', 'COMP230008', 'COMP930003']))
                                     & (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona'])) ][['persona']].drop_duplicates('persona').copy() #personaID que cursaron en YEAR
cursaronYEAR

In [ ]:
preYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['anio_academico'] < year][['persona']].drop_duplicates('persona') #personasID con inscripcion  pre-YEAR
camadaYEAR = cursaronYEAR.drop( cursaronYEAR[cursaronYEAR['persona'].isin( list(preYEAR.transpose().iloc[0].values))].index ) #camadaYEAR = cursaronYEAR SIN los que se han inscripto pre-YEAR
camadaYEAR

In [ ]:
#df_cursadas_por_alumno[(df_cursadas_por_alumno['persona'].isin(list(camadaYEAR.transpose().iloc[0].values)))]['persona'].copy().drop_duplicates() 

___________________________________________________________________________________________
### filtro ahora las cursadas de las personas que empezaron en 1C 2022

In [ ]:
dic_materias = df_lista_de_materias[(df_lista_de_materias['codigo_materia'].str.contains('LCDA21')) | (df_lista_de_materias['codigo_materia'].isin(["COMP930003","COMP930004","COMP930005", "COMP930001","COMP230008", "COMP230001", "COMP230015"]))].sort_values(by="codigo_materia")[:17].drop_duplicates().reset_index().drop(columns=('index'))
dic_materias

In [ ]:
dic_materias = dic_materias.sort_values('codigo_materia', ascending= False).reset_index()
dic_materias['index'] = list(range(13)) + [8,10,12,11]
dic_materias

In [ ]:
cursadas_camadaYEAR = df_cursadas_por_alumno[ (df_cursadas_por_alumno['persona'].isin(list(camadaYEAR.transpose().iloc[0].values)))
                                            & (df_cursadas_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) 
                                            & (df_cursadas_por_alumno['resultado'] != 'U')]
cursadas_camadaYEAR

#### Acoté con filtros la cantidad de materias (que almenos alguien la haya aprobado y que sea del plan Compu) ((y que sea más chico el vector [0,0,0...0,0,0] ))

In [ ]:
cursadas_camadaYEAR.insert(len(cursadas_camadaYEAR.columns), 'vector', [[0]*(dic_materias['index'].max()+1) for i in range(len(cursadas_camadaYEAR)) ]  )
cursadas_camadaYEAR = cursadas_camadaYEAR.sort_values(['persona', 'anio_academico', 'periodo' ] ) # hago sort 
cursadas_camadaYEAR

#### Genero los vectores (aprovechando que esta ordenado por ['persona', 'anio_academico', 'periodo']) nos permite recorrer las filas en ese orden:
 - si es persona que ya hemos visto: actualizo vector con el de la fila anterior (si es persona nueva, dejo vector de ceros)
 - y si resultado fue Aprobado, actualizo pos correspondiente al dic de materias

In [ ]:
for filaNro in range(1, len(cursadas_camadaYEAR)):
    oldFila = cursadas_camadaYEAR.iloc[filaNro-1]
    fila = cursadas_camadaYEAR.iloc[filaNro]

    if int(oldFila.persona) == int(fila.persona):
        cursadas_camadaYEAR.at[int(fila.name), 'vector']  = oldFila.vector

    if fila.resultado == 'A':
        newVect = cursadas_camadaYEAR.loc[int(fila.name), 'vector'].copy()
        k = int(dic_materias[dic_materias['codigo_materia'] == fila.codigo_materia ]['index'])
        newVect[k] = 1
        cursadas_camadaYEAR.at[int(fila.name), 'vector'] = newVect


# cursadas_camadaYEAR

In [ ]:
#sigo teniendo las mismas 94 personas que empezaron en 1C 2021?. Nop, tengo 82 (la diferencia quedó ausente)
cursadas_camadaYEAR['persona'].copy().drop_duplicates() 

agrego copia de fila 'vector' pero como string

In [ ]:
cursadas_camadaYEAR['vectStr'] = cursadas_camadaYEAR['vector'].apply(str) 
# cursadas_camadaYEAR

### Segunda pasada sobre cursadas_camadaYEAR para poder hacer los edges para cada personaID
- si es persona nueva: en caso de que NO haya aprobado, hacer loop en nodo
- si es persona de fila anterior y aprobó => hacer edge entre vectorPrevio y vectorActual 

___________________________
# Idea:
#### for persona_unique_ID, agregar los edges d esa persona. Considerando ++ el de vectZeros y el vect final 

In [ ]:
#Todas las materias de verano, tienen de nombre "1er Bimestre" y viceversa
cursadas_camadaYEAR[~(cursadas_camadaYEAR['periodo'] == "1er Bimestre") &  ( cursadas_camadaYEAR['periodo_lectivo'].str.contains("erano") )] 

In [ ]:
vectorCeroStandar = str([0]*(dic_materias['index'].max()+1))
edges = []
persona = []
esLoop = []

#asocia nodo <-> numero
dic_node = cursadas_camadaYEAR[['vectStr', 'vector']].copy().drop_duplicates('vectStr').reset_index().drop(columns=('index')) # index es el nombre del nodo que usaré para nodos de grafic
if(len(dic_node[dic_node['vectStr'] == vectorCeroStandar]) == 0):
    dic_node.loc[-1] = [vectorCeroStandar, [0]*(dic_materias['index'].max()+1) ]
    dic_node.index = dic_node.index + 1

for p in cursadas_camadaYEAR['persona'].drop_duplicates().tolist():
    personaCursadas = cursadas_camadaYEAR[cursadas_camadaYEAR['persona'] == p]

    vectA = vectorCeroStandar

    oldFila = personaCursadas.iloc[0]
    oldPeriodo = oldFila.periodo#[0]
    oldYear = oldFila.anio_academico

    
    for i in range(0,len(personaCursadas)): 
        fila = personaCursadas.iloc[i]
        periodo = fila.periodo#[0]
        year = fila.anio_academico

        if not (oldPeriodo==periodo and oldYear==year): 
            originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
            endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)

            edges.append((originPoint, endPoint))
            persona.append(int(fila.persona))
            esLoop.append(originPoint==endPoint)

            vectA = oldFila.vectStr


        oldFila = fila
        oldPeriodo = periodo
        oldYear = year

        isAproved = fila.resultado


    if vectA!=oldFila.vectStr and isAproved=='A':
        originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
        endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)
    
        # print((originPoint, endPoint))
        edges.append((originPoint, endPoint))
        persona.append(int(fila.persona))
        esLoop.append(originPoint==endPoint)
    
        
# edges

In [ ]:
# dic_node['vectStr'].tolist()

In [ ]:
# cursadas_camadaYEAR['vectStr'].tolist()

### DF final para poder hacer el grafico: agrego cantidad de apariciones para grosor de edges

In [ ]:
allEdges = pd.DataFrame({'edges' : edges, 'persona': persona, 'esLoop': esLoop})

#Loops 
losLoops = allEdges[allEdges['esLoop']].drop_duplicates().copy()
losEdges = allEdges[~(allEdges['esLoop'])]

allAllEdges = pd.concat([losEdges,losLoops])


df_edges = pd.DataFrame({'edge': allAllEdges['edges'].drop_duplicates()  })
df_edges.insert(1, 'appears', [ len(allAllEdges[allAllEdges['edges'] == df_edges['edge'].iloc[i]]) for i in range(len(df_edges)) ] ) 
df_edges

### Hacer grafico

In [ ]:
materiasCatalogo = pd.merge(dic_materias, df_lista_de_materias, on='codigo_materia', how='left')
materiasCatalogo['codigo'] = ['Intr.MC','Intr.IOO','Intr.ECdD','Proba','ALC', 'Avanzado', 'LaboDatos','Analisis2', 'Algebra','Analisis1','Algo3','Algo2','Algo1','Algebra','Algo3','Algo1', 'Algo2']
materiasCatalogo = materiasCatalogo[:13]

def strMateriasAprobadas(l): #l <- nodosCaminos.iloc[i]['vector']
    boolean_mask = [bool(valor) for valor in l]
    # temp= pd.merge(dic_materias[boolean_mask], df_lista_de_materias, on='codigo_materia', how='left')
    output = f""
    for i in range(len(materiasCatalogo[boolean_mask])): #  materiasCatalogo[boolean_mask]['codigo'].tolist():
        output+= str(materiasCatalogo[boolean_mask].iloc[i].name+1) +'. '+ materiasCatalogo[boolean_mask].iloc[i].codigo +'\n'
    return output[:len(output)-1]


________________________
### testear a mano la cantidad de gente desde vect 0 a alg, analisis o ambas
1. creo los vec

In [ ]:
vect_alg = [0]*len(materiasCatalogo)
vect_alg[0] = 1
vect_alg = str( dic_node.iloc[1].vectStr)#vect_alg)
print("algebra vect: " + vect_alg)

vect_analisis = [0]*len(materiasCatalogo)
vect_analisis[4] = 1
vect_analisis = str(vect_analisis)
print("analisis vect " + vect_analisis)

vect_both = [0]*len(materiasCatalogo)
vect_both[4] = 1
vect_both[0] = 1
vect_both = str(vect_both)
print("both vect:    " + vect_both)

In [ ]:
# print("alg " + str(len(cursadas_camadaYEAR[(cursadas_camadaYEAR['vectStr'] == vect_alg) & (cursadas_camadaYEAR['resultado'] == 'A') & (cursadas_camadaYEAR['codigo_materia'] == 'COMP930001')])))
# print("analisis " + str(len(cursadas_camadaYEAR[(cursadas_camadaYEAR['vectStr'] == vect_analisis) & (cursadas_camadaYEAR['resultado'] == 'A') & (cursadas_camadaYEAR['codigo_materia'] == 'COMP930007')] )))
# print("both pero sin las de arriba " + str(len(cursadas_camadaYEAR[(cursadas_camadaYEAR['vectStr'] == vect_both) & (cursadas_camadaYEAR['resultado'] == 'A') & (cursadas_camadaYEAR['codigo_materia'].isin(['COMP930001', 'COMP930007'])) ])))

### Gente que segun el codigo para hacer el grafico aprobaron: sólo algebra, sólo analisis, o ambas

In [ ]:
df_edges[df_edges['edge'].isin([(0,1), (0,45), (0,2)])]

### Testeo a mano de cuantas personas aprobaron primero: sólo algebra, sólo analisis, o ambas

In [ ]:
genteConAlgunaPrimero = cursadas_camadaYEAR[(cursadas_camadaYEAR['vectStr'].isin([vect_alg, vect_analisis]) ) & (cursadas_camadaYEAR['resultado'] == 'A') & (cursadas_camadaYEAR['codigo_materia'].isin(['COMP930001', 'COMP930007']))]['persona'].drop_duplicates().tolist()
curs = cursadas_camadaYEAR[cursadas_camadaYEAR['persona'].isin(genteConAlgunaPrimero)] 
curs.reset_index(inplace=True)

personasAprobaronAmbas = []
genteConSoloAlg = []
genteConSoloAnalisis = []

eskipeados = []
for p in genteConAlgunaPrimero:
    try:
        i = int(curs[(curs['persona'] == p) & (curs['resultado'] == 'A') & (curs['codigo_materia'].isin(['COMP930001', 'COMP930007'])) ].iloc[0].name)
        j = int(curs[(curs['persona'] == p) & (curs['resultado'] == 'A') & (curs['vectStr'] == vect_both)].iloc[0].name)

        if(curs.iloc[i].anio_academico == curs.iloc[j].anio_academico and curs.iloc[i].periodo[0] == curs.iloc[j].periodo[0]  ):
            personasAprobaronAmbas.append(p)
        elif curs.iloc[i].codigo_materia == 'COMP930001':
            genteConSoloAlg.append(p)
        else:
            genteConSoloAnalisis.append(p)

    except:
        eskipeados.append(p)

        largo = len(curs[(curs['persona'] == p) & (curs['resultado'] == 'A') & (curs['periodo'] == curs.iloc[i].periodo) & (curs['anio_academico'] == curs.iloc[i].anio_academico) ])
        
        if largo == 1: #gente que en ese cuatri aprobó solo 1 materia (algebra o analisis)
            if curs.iloc[i].codigo_materia == 'COMP930001': #algebra
                genteConSoloAlg.append(p)
            else:
                genteConSoloAnalisis.append(p)

        pass


In [ ]:
# len(
curs[(curs['persona'] == 20095) & (curs['anio_academico'] == 2015)  & (curs['periodo'] == '1er Cuatrimestre')   &  (curs['resultado'] == 'A')  ]

In [ ]:
print(len(personasAprobaronAmbas))
print(len(genteConSoloAlg))
print(len(genteConSoloAnalisis))
#genteConSoloAnalisis

In [ ]:
print([x for x in personasAprobaronAmbas if x not in allEdges[allEdges['edges'] == (0,2)]['persona'].tolist() ])
print([x for x in genteConSoloAlg if x not in allEdges[allEdges['edges'] == (0,1)]['persona'].tolist() ])
print([x for x in genteConSoloAnalisis if x not in allEdges[allEdges['edges'] == (0,45)]['persona'].tolist() ])

im finished that thing

_______________________
### Continuo grafico

In [ ]:
# # nodos que solo les entra flecha de weight 1 ó 2
# nodeExclude = set(df_edges[df_edges['appears'] >= 3 ]['edge'].explode()) - set(df_edges[(df_edges['appears'] <= 2) ]['edge'].explode()) 
# nodeExclude #lo d abajo

##### sugerencia de Mati: limitar a nodos con aristas de >= 3 personas

In [ ]:
print(len(df_edges))
df_edges = df_edges[df_edges['appears'] >= 3 ]
len(df_edges)

In [ ]:
tpl = set()
for tuple in df_edges['edge'].tolist():
    tpl.add(tuple[0])
    tpl.add(tuple[1])
#tpl son los index de los nodos que SI les llega o sale una edge de ellos                 # tpl = tpl - nodeExclude
tpl = list(tpl)

In [ ]:
#materiasCatalogo
#strMateriasAprobadas([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

#Plan del 2022
nodosDelPlan = [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0], #algebra y analisis
                [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], #algo1, algebra y analisis
                [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1], #analisis2, alc, labodatos algo1, algebra y analisis
                [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1], #algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], #algo3, proba algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], #algo3, proba algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                ]

In [ ]:
dot = Digraph()
for i in tpl: #range(len(dic_node)):
    nombre =  strMateriasAprobadas(dic_node.loc[i]['vector'])

    if(dic_node.loc[i]['vector'] in nodosDelPlan):
        elColor="#62b3ff"
    else:
        elColor="#000000"
    
    dot.node(str(i), str(i) +'\n' + nombre, color = elColor)

for i in range(len(df_edges)):
    beginning = df_edges['edge'].iloc[i][0]
    end = df_edges['edge'].iloc[i][1]

    width = df_edges['appears'].iloc[i]

    if((dic_node.loc[beginning]['vector'] in nodosDelPlan) and 
       (dic_node.loc[end]['vector'] in nodosDelPlan) and 
       abs(nodosDelPlan.index(dic_node.loc[beginning]['vector']) - nodosDelPlan.index(dic_node.loc[end]['vector'])) == 1):
        elColor="#62b3ff"
    else:
        elColor="#5c5c5c"
    dot.edge(str(beginning), str(end), penwidth = str( width/4 ), label=str( width ), color=elColor)


dot

In [ ]:
nodosInteresantes=[135]
nodosCaminos = dic_node[dic_node.index.isin(nodosInteresantes)]
nodosCaminos

In [ ]:
for i in range(len(nodosCaminos)):
    # print("nodo "+str(nodosCaminos.iloc[i].name))
    # boolean_mask = [bool(valor) for valor in nodosCaminos.iloc[i]['vector']]
    # dic_materias[boolean_mask]

    # print(pd.merge(dic_materias[boolean_mask], df_lista_de_materias, on='codigo_materia', how='left')['nombre_materia'].tolist())
    print(strMateriasAprobadas(nodosCaminos.iloc[i]['vector']))
    print("______________________")

In [ ]:
# path = "hola.xlsx"
# dftest = pd.read_excel(path, sheet_name="finales por alumno (2)")

# dftest[(dftest['instancia'] != 3)& (dftest['instancia'] != 8) & (dftest['resultado'] == "A")].sort_values(by='instancia')